# Climate-GEA — SNP vs non-SNP vs SV: significant peaks & genes GAINED per layer

clq0.9-block WZA climate-GEA (gen9, **bio1**), three models (binomial / Kendall / LFMM)
x three variant classes (SNP / non-SNP = SV+indel / SV-only). Blocks share one clq0.9
partition across classes, so a block significant in non-SNP or SV but **not** in the SNP
scan is a genuine spatial gain from that layer.

- **Table 1 / Table 2** — per model: `n_snp`/`n_nonsnp`/`n_sv` significant blocks, and the
  gain columns `n_nonsnp_not_snp` / `n_sv_not_snp` (+ `pct` = fraction of that class's own
  hits that SNPs miss), at Bonferroni (0.05/n) and BH-FDR (q<0.05).
- **Gene tables** — the genes under non-SNP-only / SV-only significant blocks (union across
  models), annotated, with which models flagged them.

Reading guide (established in this project): binomial & Kendall are the **inflated raw
references** (pool pseudoreplication); **LFMM is the calibrated model**. Weight the LFMM
row / LFMM-flagged genes accordingly.

In [1]:

import os, sys
import numpy as np, pandas as pd
os.chdir("/global/scratch/users/tbellg/kmate"); sys.path.insert(0, "analysis/grenenet_gea")
sys.path.insert(0, "analysis/grenenet_gea/gea_newpanel")
import lib, blocks_clq09
GNP = "analysis/grenenet_gea/gea_newpanel/results"
MODELS = ["binomial", "kendall", "lfmm"]; CLASSES = ["snp", "nonsnp", "sv"]

def load_wza(model, cls):
    w = pd.read_csv(f"{GNP}/wza_{model}_{cls}_clq09.csv").rename(columns={"index": "block"})
    w["block"] = w["block"].astype(str)
    return w[w["Z_pVal"].notna()].copy()

def sig_sets(model, cls, q=0.05):
    w = load_wza(model, cls); p = w["Z_pVal"].to_numpy()
    bonf = set(w.loc[p < 0.05 / len(w), "block"])
    qv = lib.bh(p)
    fdr = set(w.loc[qv < q, "block"])
    return bonf, fdr

# per (model, class) -> (bonf_set, fdr_set)
S = {(m, c): sig_sets(m, c) for m in MODELS for c in CLASSES}
print("WZA blocks per class:", {c: len(load_wza("lfmm", c)) for c in CLASSES})


WZA blocks per class: {'snp': 81699, 'nonsnp': 68780, 'sv': 11571}


## Tables 1 & 2 — significant blocks per class + gain from non-SNP / SV

In [2]:

def gain_table(thr):   # thr: 0=Bonferroni, 1=FDR
    rows = []
    for m in MODELS:
        s = {c: S[(m, c)][thr] for c in CLASSES}
        row = {"model": m, "n_snp": len(s["snp"]), "n_nonsnp": len(s["nonsnp"]), "n_sv": len(s["sv"])}
        for b in ("nonsnp", "sv"):
            new = s[b] - s["snp"]
            row[f"n_{b}_not_snp"] = len(new)
            row[f"pct_{b}_not_snp"] = round(len(new) / len(s[b]), 3) if s[b] else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

gea_bonf_df = gain_table(0)
gea_fdr_df = gain_table(1)
gea_bonf_df.to_csv(f"{GNP}/snp_vs_nonsnp/gea_classgain_bonf.csv", index=False)
gea_fdr_df.to_csv(f"{GNP}/snp_vs_nonsnp/gea_classgain_fdr.csv", index=False)
print("built gain tables")


built gain tables


### Table 1 — Bonferroni (0.05) significant GEA peaks, per model (bio1, gen9)

In [3]:
gea_bonf_df


,model,n_snp,n_nonsnp,n_sv,n_nonsnp_not_snp,pct_nonsnp_not_snp,n_sv_not_snp,pct_sv_not_snp
0,binomial,3,5,0,4,0.800,0,NaN
1,kendall,3,4,0,2,0.500,0,NaN
2,lfmm,13,11,4,5,0.455,4,1.0


### Table 2 — BH-FDR (q<0.05) significant GEA peaks, per model (bio1, gen9)

In [4]:
gea_fdr_df


,model,n_snp,n_nonsnp,n_sv,n_nonsnp_not_snp,pct_nonsnp_not_snp,n_sv_not_snp,pct_sv_not_snp
0,binomial,11,34,2,27,0.794,2,1.0
1,kendall,19,19,0,14,0.737,0,NaN
2,lfmm,66,52,5,33,0.635,5,1.0


## Genes under non-SNP-only / SV-only GEA peaks

For every clq0.9 block a class (non-SNP or SV) flags significant that the SNP scan does
not — in ANY model — map the block interval (+/-2 kb flank) to TAIR10 genes and record
which models flagged it. `klass` = layer, `overlap` = in_block vs flank2kb, `n_models` /
`models` = where the layer fired but SNPs didn't. Two thresholds (Bonferroni, FDR).
Subject to the per-locus null; hypothesis-generating.

In [5]:

FLANK = 2000
bmap = {r.block: (r.chrom, int(r.start_pos), int(r.end_pos)) for r in blocks_clq09.load_blocks().itertuples()}
genes = lib.load_genes()

# optional descriptions cache (offline-safe; no live Ensembl call)
DESC = "analysis/grenenet_gea/varexp/gene_descriptions.csv"
desc = {}
if os.path.exists(DESC):
    _d = pd.read_csv(DESC).fillna("")
    desc = {r.gene: (getattr(r, "symbol", ""), getattr(r, "function", "")) for r in _d.itertuples()}

def gene_table(thr):
    hits = {}
    for cls in ("nonsnp", "sv"):
        blk2models = {}
        for m in MODELS:
            only = S[(m, cls)][thr] - S[(m, "snp")][thr]
            for b in only:
                blk2models.setdefault(b, set()).add(m)
        for b, ms in blk2models.items():
            if b not in bmap:
                continue
            c, s0, e0 = bmap[b]
            inb = genes[(genes.chrom == c) & (genes.start <= e0) & (genes.end >= s0)]
            flk = genes[(genes.chrom == c) & (genes.start <= e0 + FLANK) & (genes.end >= s0 - FLANK)]
            flk = flk[~flk.gene.isin(inb.gene)]
            for gid, nm, otype in ([(g, n, "in_block") for g, n in zip(inb.gene, inb.name.fillna(""))]
                                   + [(g, n, "flank2kb") for g, n in zip(flk.gene, flk.name.fillna(""))]):
                d = hits.setdefault((gid, cls), dict(gene=gid, name=nm, klass=cls, chrom=c,
                                                     overlap=otype, blocks=set(), models=set()))
                d["blocks"].add(b); d["models"].update(ms)
                if otype == "in_block":
                    d["overlap"] = "in_block"
    rows = [dict(gene=d["gene"], name=d["name"],
                 symbol=desc.get(d["gene"], ("", ""))[0], function=desc.get(d["gene"], ("", ""))[1],
                 klass=d["klass"], chrom=d["chrom"], overlap=d["overlap"],
                 n_blocks=len(d["blocks"]), n_models=len(d["models"]),
                 models=";".join(sorted(d["models"]))) for d in hits.values()]
    df = pd.DataFrame(rows)
    if len(df):
        df = df.sort_values(["klass", "n_models", "overlap", "gene"],
                            ascending=[True, False, True, True]).reset_index(drop=True)
    return df

genes_bonf_df = gene_table(0)
genes_fdr_df = gene_table(1)
genes_bonf_df.to_csv(f"{GNP}/snp_vs_nonsnp/gea_gained_genes_bonf.csv", index=False)
genes_fdr_df.to_csv(f"{GNP}/snp_vs_nonsnp/gea_gained_genes_fdr.csv", index=False)
print(f"gained genes — Bonferroni: {len(genes_bonf_df)}   FDR: {len(genes_fdr_df)}")


gained genes — Bonferroni: 26   FDR: 140


### Genes — Bonferroni (non-SNP-only / SV-only GEA peaks)

In [6]:
genes_bonf_df


,gene,name,symbol,function,klass,chrom,overlap,n_blocks,n_models,models
0,AT1G72510,AT1G72510,,,nonsnp,Chr1,flank2kb,1,1,binomial
1,AT1G72520,AT1G72520,,,nonsnp,Chr1,flank2kb,1,1,binomial
2,AT3G06350,AT3G06350,,,nonsnp,Chr3,flank2kb,1,1,binomial
3,AT4G25550,AT4G25550,,,nonsnp,Chr4,flank2kb,1,1,binomial
4,AT4G25560,AT4G25560,,,nonsnp,Chr4,flank2kb,1,1,binomial
5,AT5G24110,AT5G24110,,,nonsnp,Chr5,flank2kb,1,1,binomial
6,AT5G24130,AT5G24130,,,nonsnp,Chr5,flank2kb,1,1,binomial
7,AT5G24230,AT5G24230,,,nonsnp,Chr5,flank2kb,1,1,kendall
8,AT5G24250,AT5G24250,,,nonsnp,Chr5,flank2kb,1,1,kendall
9,AT5G65270,AT5G65270,,,nonsnp,Chr5,flank2kb,1,1,lfmm


### Genes — BH-FDR (non-SNP-only / SV-only GEA peaks)

In [7]:
genes_fdr_df


,gene,name,symbol,function,klass,chrom,overlap,n_blocks,n_models,models
0,AT1G16090,AT1G16090,,,nonsnp,Chr1,flank2kb,1,2,kendall;lfmm
1,AT5G24230,AT5G24230,,,nonsnp,Chr5,flank2kb,1,2,kendall;lfmm
2,AT1G12970,AT1G12970,,,nonsnp,Chr1,in_block,1,2,binomial;kendall
3,AT1G16110,AT1G16110,,,nonsnp,Chr1,in_block,1,2,kendall;lfmm
4,AT1G22760,AT1G22760,PAB3,poly(A) binding protein 3,nonsnp,Chr1,in_block,1,2,kendall;lfmm
...,...,...,...,...,...,...,...,...,...,...
135,AT5G28220,AT5G28220,,,sv,Chr5,in_block,1,1,lfmm
136,AT5G28235,AT5G28235,,,sv,Chr5,in_block,1,1,lfmm
137,AT5G28237,AT5G28237,,,sv,Chr5,in_block,1,1,lfmm
138,AT5G38020,AT5G38020,,,sv,Chr5,in_block,1,1,lfmm


## Bottom line
- `n_nonsnp_not_snp` / `n_sv_not_snp` = the blocks the non-SNP / SV layer adds beyond SNPs.
  In LFMM (the calibrated model) these are the defensible gains; binomial/Kendall gains are
  inflated by pseudoreplication and should be read as an upper bound.
- The gene tables list what those gained blocks contain — hypothesis-generating, per-locus
  null applies. This is the bio1 view; the per-axis extension (bio5/12/13/16/19/pc1) is the
  same tables with one row per (axis, model).